<a href="https://colab.research.google.com/github/sanjoggrg1/Sanjog-TECH405/blob/main/Sanjog_WK5_TECH405.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense

In [3]:
# Here are Parameters
VOCAB_SIZE = 1000 # top 10,000 most common words
MAX_LENGTH = 200 # caps 200 words in a review

In [4]:
# Now Loading the data and splitting into train  and test
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [6]:
# Padding sequences to make the input uniform
X_train_padded = pad_sequences(X_train, maxlen=MAX_LENGTH, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test, maxlen=MAX_LENGTH, padding='post', truncating='post')

print("Training Data shape")
print(X_train.shape)

Training Data shape
(25000,)


In [8]:
# sequential model
model = Sequential()

# Embedding Layer and converts the word integer indices into dense vectors of fixed size (128 dimensions)
model.add(Embedding(input_dim=VOCAB_SIZE, output_dim=128, input_length=MAX_LENGTH))

model.add(Bidirectional(LSTM(64)))

# sigmoid for binary classificaiton
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [6]:
model.build(input_shape=(None, MAX_LENGTH))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │       128,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 226,945 (886.50 KB)

 Trainable params: 226,945 (886.50 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# Training a model
history = model.fit(
    X_train_padded,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2 # 20% of training data to check accuracy after each epoch
)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.7025 - loss: 0.5641 - val_accuracy: 0.8072 - val_loss: 0.4573
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.8126 - loss: 0.4279 - val_accuracy: 0.8352 - val_loss: 0.4112
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.8315 - loss: 0.3938 - val_accuracy: 0.8306 - val_loss: 0.3859
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.8446 - loss: 0.3704 - val_accuracy: 0.8276 - val_loss: 0.3920
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.8508 - loss: 0.3538 - val_accuracy: 0.8358 - val_loss: 0.3873
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.8688 - loss: 0.3194 - val_accuracy: 0.8346 - val_loss: 0.3801
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.8787 - loss: 0.2995 - val_accuracy: 0.8120 - val_loss: 0.4302
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.8812 - loss: 0.2918 - val_ac

In [10]:
test_loss, test_accuracy = model.evaluate(X_test_padded, y_test)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.8370 - loss: 0.3957
Test Accuracy: 83.70%


In [11]:
# Checking with own texts
word_index = imdb.get_word_index()

reviews = ["loved the movie", "this movie was a waste of time", "i will watch this movie again", "it has become my favorite movie", "i am dissapointed i had high expectations"]
for review in reviews:
    words = review.split()
    tokenized_review = []
    for word in words:
        if word in word_index and (word_index[word] + 3) < VOCAB_SIZE:
            tokenized_review.append(word_index[word] + 3)
        else:
            tokenized_review.append(2) # 2 is given to those words which are rare

    padded_review = pad_sequences([tokenized_review], maxlen=MAX_LENGTH, padding='post', truncating='post')
    prediction = model.predict(padded_review, verbose=0)[0][0]
    sentiment = "Positive" if prediction >= 0.5 else "Negative"
    print(f"Review: {review}\nSentiment:{sentiment}\n\n")

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
Review: loved the movie
Sentiment:Positive


Review: this movie was a waste of time
Sentiment:Negative


Review: i will watch this movie again
Sentiment:Positive


Review: it has become my favorite movie
Sentiment:Positive


Review: i am dissapointed i had high expectations
Sentiment:Negative


